# VULCAN-JAX Quickstart

Install:
```
pip install -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ vulcan-jax
```

In [ ]:
import vulcan_jax

print("vulcan-jax version:", vulcan_jax.__version__)

## Run HD189733b to convergence

The default config ships with the package -- atmosphere, network, and photo cross-sections are all included.

In [ ]:
cfg = vulcan_jax.make_config(use_print_prog=False)

In [ ]:
import time

# Build the initial state
rs = vulcan_jax.RunState.with_pre_loop_setup(cfg)

# Run the integration to convergence. A make_config() cfg must be passed to
# BOTH the runner and the writer -- constructing them bare would silently
# run on the package defaults instead of the overrides above.
from vulcan_jax import outer_loop, op_jax, legacy_io

solver = op_jax.Ros2JAX()
output = legacy_io.Output(cfg=cfg)
integ = outer_loop.OuterLoop(solver, output, cfg=cfg)

t0 = time.time()
rs = integ(rs)
wall = time.time() - t0

end_case = int(rs.params.end_case)
labels = {1: "converged", 2: "runtime cap", 3: "count_max cap"}
print(
    f"Done: {labels.get(end_case, end_case)}, {int(rs.params.count)} steps, {wall:.1f}s"
)

In [ ]:
import numpy as np

# Inspect the converged state
print("Number of layers:", rs.atm.Tco.shape[0])
print("Number of species:", rs.step.y.shape[1])
print("Species list:", vulcan_jax.chem_funs.spec_list[:10], "...")

## Plot converged volume mixing ratios

VMR profiles for key species vs pressure at the converged photochemical steady state.

In [ ]:
import matplotlib.pyplot as plt

species_to_plot = ["H2O", "CO", "CO2", "CH4", "NH3", "HCN"]
spec_list = vulcan_jax.chem_funs.spec_list

pressure_bar = np.asarray(rs.atm.pco) / 1e6
ymix = np.asarray(rs.step.ymix)

fig, ax = plt.subplots(figsize=(6, 5))
for sp in species_to_plot:
    idx = spec_list.index(sp)
    ax.plot(ymix[:, idx], pressure_bar, label=sp)

ax.set_xscale("log")
ax.set_yscale("log")
ax.invert_yaxis()
ax.set_xlabel("Volume Mixing Ratio")
ax.set_ylabel("Pressure (bar)")
ax.set_title("HD 189733b: Converged Abundances")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

## Access individual modules

All VULCAN-JAX modules are importable under the `vulcan_jax` namespace.

In [ ]:
from vulcan_jax import chem_funs

print("Network:", chem_funs.ni, "species,", chem_funs.nr, "reactions")
print("First 10 species:", chem_funs.spec_list[:10])